# 3. Retrieval & Generation - RAG Pipeline with GPT-4o-mini (v3: Chain-of-Thought)

This notebook implements the core RAG (Retrieval-Augmented Generation) pipeline for medical question answering. It loads the ChromaDB vector index built in Notebook 2, retrieves relevant document chunks for each query, reranks them using a cross-encoder model, and generates answers using GPT-4o-mini with **Chain-of-Thought (CoT) prompting**.

**v3 improvements:**
- Chain-of-Thought reasoning before the final answer (improves RAGAS Faithfulness)
- Full reasoning saved in results for downstream evaluation
- `temperature=0` for deterministic responses, `max_tokens=300` for reasoning

In [1]:
import json
import os
import re
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv

import chromadb
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.llms import ChatMessage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore

In [2]:
load_dotenv(Path("..") / ".env")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
CHROMA_DIR = BASE_DIR / "chroma_db"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

## 3.1 Load index and configure models

In [3]:
# Embedding model (same as indexing)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.embed_model = embed_model

# LLM: temperature=0 for deterministic CoT, max_tokens=300 for reasoning
llm = OpenAI(model="gpt-4o-mini", max_tokens=300, temperature=0)
Settings.llm = llm

print("Models configured (temperature=0, max_tokens=300 for CoT reasoning)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models configured (temperature=0, max_tokens=300 for CoT reasoning)


In [4]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
chroma_collection = chroma_client.get_or_create_collection("pubmed_rag")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_vector_store(vector_store)
print(f"Loaded index with {chroma_collection.count()} chunks")

Loaded index with 43806 chunks


## 3.2 Configure retriever and reranker

In [5]:
# Retriever: get top 20 by embedding similarity
retriever = index.as_retriever(similarity_top_k=20)

# Reranker: cross-encoder to rerank top 20 -> top 5
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=5
)

print("Retriever (top_k=20) + Reranker (top_n=5) configured")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever (top_k=20) + Reranker (top_n=5) configured


In [6]:
test_query = "Is anorectal endosonography valuable in dyschesia?"

# Retrieve
nodes_before_rerank = retriever.retrieve(test_query)
print(f"Before rerank: {len(nodes_before_rerank)} chunks")

# Rerank
nodes_after_rerank = reranker.postprocess_nodes(nodes_before_rerank, query_str=test_query)
print(f"After rerank: {len(nodes_after_rerank)} chunks")

for i, node in enumerate(nodes_after_rerank):
    print(f"  {i+1}. [{node.metadata.get('source_id', 'N/A')}] score={node.score:.4f}: {node.text[:100]}...")

Before rerank: 10 chunks
After rerank: 5 chunks
  1. [Document_47183] score=5.1019: Dyschesia can be provoked by inappropriate defecation movements. The aim of this prospective study w...
  2. [Document_216720] score=-0.9560: title: Dyssynergia
content: doctors. An abnormal or prolonged time of expulsion of the balloon is se...
  3. [Document_627762] score=-1.2793: title: Dyssynergia
content: a decrease in intrarectal pressure defecation can occur. Anal sphincter ...
  4. [Document_754472] score=-2.5774: title: Anismus
content: dyssynergic defecation is given as "inappropriate contraction of the pelvic ...
  5. [Document_453749] score=-3.3392: title: Medical ultrasound
content: capabilities in this area. The appendix can sometimes be seen whe...


## 3.3 Load queries

In [7]:
queries = []
with open(DATA_DIR / "pubmedqa.jsonl", "r") as f:
    for line in f:
        queries.append(json.loads(line))

print(f"Loaded {len(queries)} queries")
print(f"Example: {queries[0]['query']}")

Loaded 500 queries
Example: Is anorectal endosonography valuable in dyschesia?


## 3.4 Run RAG pipeline

In [8]:
SYSTEM_PROMPT = """You are a medical expert answering yes/no/maybe questions based on PubMed abstracts.
Rules:
- First, reason step-by-step about what the evidence says (2-3 sentences).
- Then give your final answer on a new line in the format: "Final Answer: yes" or "Final Answer: no" or "Final Answer: maybe"
- Answer "yes" if the evidence supports the claim.
- Answer "no" if the evidence contradicts or does not support the claim.
- Answer "maybe" ONLY if the evidence is genuinely mixed or insufficient to decide.
- Most questions have a definitive answer. Prefer "yes" or "no" over "maybe"."""

FEW_SHOT_EXAMPLES = """Example 1:
Question: Is increased gravitational stress beneficial for bone density?
Context: Studies show weight-bearing exercise increases bone mineral density by 2-8% in postmenopausal women...
Reasoning: The evidence indicates that weight-bearing exercise, which increases gravitational stress on bones, leads to measurable increases in bone mineral density (2-8%). This supports the claim that gravitational stress benefits bone density.
Final Answer: yes

Example 2:
Question: Does smoking cessation reduce cardiovascular risk?
Context: After 1 year of cessation, coronary heart disease risk drops by 50%. After 15 years, risk equals that of a non-smoker...
Reasoning: The evidence clearly shows that quitting smoking leads to substantial cardiovascular risk reduction - 50% within one year and full normalization after 15 years. This strongly supports the claim.
Final Answer: yes

Example 3:
Question: Is homeopathy effective for treating asthma?
Context: A systematic review of 6 RCTs found no significant difference between homeopathic treatments and placebo in lung function or symptom scores...
Reasoning: A systematic review of 6 randomized controlled trials found no significant difference between homeopathy and placebo. This is strong evidence against effectiveness, as systematic reviews of RCTs are high-quality evidence.
Final Answer: no

Example 4:
Question: Can MRI replace biopsy for diagnosing prostate cancer?
Context: MRI showed sensitivity of 91% but specificity of only 37%. While useful for risk stratification, results were inconsistent across centers...
Reasoning: While MRI has high sensitivity (91%), its low specificity (37%) means many false positives. Additionally, results are inconsistent across centers, making it unreliable as a biopsy replacement. It may complement but cannot replace biopsy.
Final Answer: maybe

"""

NUM_VOTES = 1  # Single deterministic call (temperature=0, no self-consistency needed)

def extract_answer(text):
    """Extract yes/no/maybe from LLM response, prioritizing 'Final Answer:' format."""
    text_clean = text.strip().lower()
    # First try to find "Final Answer: ..." pattern
    match = re.search(r'final answer:\s*(yes|no|maybe)\b', text_clean, re.I)
    if match:
        return match.group(1).lower()
    # Fallback: look for standalone yes/no/maybe
    if text_clean in ("yes", "no", "maybe"):
        return text_clean
    match = re.search(r'\b(yes|no|maybe)\b', text_clean, re.I)
    return match.group(1).lower() if match else "unknown"

def majority_vote(answers):
    """Return the majority answer from a list of answers."""
    from collections import Counter
    valid = [a for a in answers if a in ("yes", "no", "maybe")]
    if not valid:
        return "unknown"
    counts = Counter(valid)
    return counts.most_common(1)[0][0]

def build_prompt(query_text, contexts):
    """Build the prompt from query and retrieved contexts."""
    context_parts = []
    for i, ctx in enumerate(contexts):
        context_parts.append(f"[Document {i+1}]\n{ctx}")
    context = "\n\n".join(context_parts)
    return f"""{FEW_SHOT_EXAMPLES}Now answer this question:

Context:
{context}

Question: {query_text}

Reasoning:"""

def call_llm_with_retry(llm, prompt, max_retries=3):
    """Call LLM API with retry and exponential backoff for rate limits."""
    for attempt in range(max_retries):
        try:
            response = llm.chat([
                ChatMessage(role="system", content=SYSTEM_PROMPT),
                ChatMessage(role="user", content=prompt),
            ])
            return str(response.message.content)
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = 2 ** attempt  # 1s, 2s, 4s
                time.sleep(wait)
                continue
            raise
    # Final attempt without catching
    response = llm.chat([
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=prompt),
    ])
    return str(response.message.content)

print(f"Chain-of-Thought configured: {NUM_VOTES} call per query (temperature=0, deterministic)")

Chain-of-Thought configured: 1 call per query (temperature=0, deterministic)


In [9]:
# PHASE 1: Retrieval + Reranking (local, sequential, fast)
print("Phase 1: Retrieving and reranking (local)...")

retrieval_data = []
for i, q in enumerate(queries):
    nodes_initial = retriever.retrieve(q["query"])
    nodes_reranked = reranker.postprocess_nodes(nodes_initial, query_str=q["query"])
    
    docs_before = list(dict.fromkeys(n.metadata.get("source_id", "") for n in nodes_initial))
    docs_after = list(dict.fromkeys(n.metadata.get("source_id", "") for n in nodes_reranked))
    contexts = [node.text for node in nodes_reranked]
    prompt = build_prompt(q["query"], contexts)
    
    retrieval_data.append({
        "idx": i,
        "id": q["id"],
        "query": q["query"],
        "ground_truth": q["ground_truth"],
        "golden_doc": q["golden_doc"],
        "docs_before_rerank": docs_before,
        "docs_after_rerank": docs_after,
        "contexts": contexts,
        "prompt": prompt,
    })
    
    if (i + 1) % 100 == 0:
        print(f"  Retrieved {i + 1}/{len(queries)}...")

print(f"Phase 1 complete: {len(retrieval_data)} queries retrieved\n")

Phase 1: Retrieving and reranking (local)...


  Retrieved 100/500...


  Retrieved 200/500...


  Retrieved 300/500...


  Retrieved 400/500...


  Retrieved 500/500...
Phase 1 complete: 500 queries retrieved



In [10]:
# PHASE 2: CoT generation (parallel API calls, 1 call per query = 500 total)
NUM_WORKERS = 10
print(f"Phase 2: CoT generation with {len(retrieval_data)} queries = {len(retrieval_data)} API calls")
print(f"  Workers: {NUM_WORKERS} (temperature=0, deterministic)\n")

# Generate one CoT response per query
all_responses = {}  # {query_idx: raw_response}

gen_tasks = [(item["idx"], item["prompt"]) for item in retrieval_data]
errors = 0

def generate_single(task):
    """Generate one CoT response for a query."""
    idx, prompt = task
    answer = call_llm_with_retry(llm, prompt)
    return idx, answer

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(generate_single, task): task for task in gen_tasks}
    completed = 0
    
    for future in as_completed(futures):
        task = futures[future]
        try:
            idx, answer = future.result()
            all_responses[idx] = answer
        except Exception as e:
            idx, _ = task
            all_responses[idx] = f"ERROR: {str(e)}"
            errors += 1
        
        completed += 1
        if completed % 100 == 0:
            print(f"  Completed {completed}/{len(gen_tasks)} API calls...")

print(f"\nPhase 2 complete! {len(gen_tasks) - errors} succeeded, {errors} errors")

# Extract answers
results = []
for i, item in enumerate(retrieval_data):
    idx = item["idx"]
    raw_response = all_responses[idx]
    extracted = extract_answer(raw_response)
    
    results.append({
        "id": item["id"],
        "query": item["query"],
        "ground_truth": item["ground_truth"],
        "golden_doc": item["golden_doc"],
        "generated_answer": extracted,
        "raw_response": raw_response,  # Full CoT response for RAGAS
        "votes": [extracted],  # Single vote for compatibility
        "docs_before_rerank": item["docs_before_rerank"],
        "docs_after_rerank": item["docs_after_rerank"],
        "contexts": item["contexts"],
    })

print(f"\nTotal results: {len(results)}")

# Show example CoT response
print(f"\nExample CoT response:")
print(f"Q: {results[0]['query']}")
print(f"Response:\n{results[0]['raw_response']}")
print(f"Extracted: {results[0]['generated_answer']}")

Phase 2: CoT generation with 500 queries = 500 API calls
  Workers: 10 (temperature=0, deterministic)



Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


...

Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


Retrying llama_index.llms.openai.base.OpenAI._chat in 9 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-j9fMw3biSQkS7jYxHFBJxEiF on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}.


  Completed 500/500 API calls...

Phase 2 complete! 483 succeeded, 17 errors

Total results: 500

Example CoT response:
Q: Is anorectal endosonography valuable in dyschesia?
Response:
The context indicates that anorectal endosonography is used to demonstrate dysfunction of the anal sphincter and/or the musculus puborectalis in patients with dyschesia. This suggests that the technique is valuable for diagnosing the underlying issues related to dyschesia, as it provides important insights into the anatomical and functional aspects of the anorectal region. Therefore, the evidence supports the claim that anorectal endosonography is valuable in dyschesia.

Final Answer: yes
Extracted: yes


## 3.5 Save results

In [11]:
# Save full results as JSONL
results_path = RESULTS_DIR / "rag_results.jsonl"
with open(results_path, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"Saved {len(results)} results to {results_path}")

# Also save a summary CSV
summary = []
for r in results:
    predicted = r["generated_answer"]  # already extracted via majority vote
    summary.append({
        "id": r["id"],
        "query": r["query"],
        "ground_truth": r["ground_truth"],
        "predicted": predicted,
        "correct": predicted == r["ground_truth"].lower(),
        "votes": str(r["votes"]),
        "num_docs_retrieved": len(r["docs_after_rerank"]),
    })

df_summary = pd.DataFrame(summary)
df_summary.to_csv(RESULTS_DIR / "rag_summary.csv", index=False)

print(f"\nQuick accuracy check:")
print(f"  Accuracy: {df_summary['correct'].mean():.2%}")
print(f"\nPrediction distribution:")
print(df_summary['predicted'].value_counts())
print(f"\nGround truth distribution:")
print(df_summary['ground_truth'].value_counts())

Saved 500 results to ../results/rag_results.jsonl

Quick accuracy check:
  Accuracy: 46.20%

Prediction distribution:
predicted
yes        190
maybe      179
no         114
unknown     17
Name: count, dtype: int64

Ground truth distribution:
ground_truth
yes      276
no       169
maybe     55
Name: count, dtype: int64


In [12]:
print("=== Spot Check: 5 Examples (with CoT reasoning) ===\n")
for r in results[:5]:
    print(f"ID: {r['id']}")
    print(f"Q: {r['query']}")
    print(f"Ground Truth: {r['ground_truth']}")
    print(f"Final Answer: {r['generated_answer']}")
    print(f"CoT Response:\n{r['raw_response'][:400]}")
    print(f"Docs retrieved: {r['docs_after_rerank'][:3]}...")
    print("-" * 80)

=== Spot Check: 5 Examples (with CoT reasoning) ===

ID: pubmed_1
Q: Is anorectal endosonography valuable in dyschesia?
Ground Truth: yes
Final Answer: yes
CoT Response:
The context indicates that anorectal endosonography is used to demonstrate dysfunction of the anal sphincter and/or the musculus puborectalis in patients with dyschesia. This suggests that the technique is valuable for diagnosing the underlying issues related to dyschesia, as it provides important insights into the anatomical and functional aspects of the anorectal region. Therefore, the evidence 
Docs retrieved: ['Document_47183', 'Document_216720', 'Document_627762']...
--------------------------------------------------------------------------------
ID: pubmed_2
Q: Is there a connection between sublingual varices and hypertension?
Ground Truth: yes
Final Answer: yes
CoT Response:
The evidence from Document 1 indicates a significant association between sublingual varices and hypertension, with an odds ratio of 2.25 an

**v3 Results saved:**
- `results/rag_results.jsonl` — Full results with CoT reasoning (`raw_response` field), votes, and retrieval data
- `results/rag_summary.csv` — Summary with accuracy per query

The `raw_response` field contains the full Chain-of-Thought reasoning, which RAGAS uses for Faithfulness evaluation. The next notebook (04) evaluates retrieval quality, runs ablation studies, and evaluates answer quality with RAGAS.